# VoD 위험도 분류 리팩토링: 누설 진단 + 개선 비교 실험

이 노트북은 `16_vod_radar_lidar_end_to_end_presentation.ipynb`의 위험도 분류 구조를 다음 목표로 리팩토링한 버전입니다.

## 최종 목표
1. 라벨 누설(label leakage) 제거
2. 프레임/시계열 기반 독립 분할 적용
3. Rule-based 위험도와 ML-based 위험도 예측 분리
4. 단순 모델 vs 비교적 고급 모델 비교
5. `accuracy 1.000`의 원인을 실험적으로 설명
6. 발표/PPT에 바로 넣을 수 있는 한국어 해석 제공

## 문제 정의 및 실험 설계

현재 구조의 핵심 문제:
- `risk_label` 생성 규칙(TTC/거리/속도)과 모델 입력 특징이 직접 중복됨
- random split에서 인접 프레임의 유사 샘플이 train/test에 함께 들어감

따라서 이 노트북은 아래 비교를 반드시 수행합니다.
- **Feature Set C (진단용, 누설 포함)** vs **Feature Set A/B (anti-leakage)**
- **random split** vs **group(frame) split** vs **time split**
- **LogisticRegression / RandomForest / HistGradientBoosting / SVM(RBF)**

> 주의: ML 타깃은 외부 GT가 아니라 rule 기반 pseudo-label입니다. 따라서 ML 결과는 운영 로직 대체가 아니라 보조 추론 성능 확인 실험입니다.

## 0. 공통 Import + 한글 폰트 설정

In [ ]:
from __future__ import annotations

import json
import logging
import os
import random
import sys
import warnings
from pathlib import Path
from typing import Any

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.cluster import DBSCAN
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight

NOTEBOOK_DIR = Path.cwd().resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

import bev_lidar_detector_train as bev

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

FONT_PATH = "C:/Windows/Fonts/malgun.ttf"
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
if Path(FONT_PATH).is_file():
    font_manager.fontManager.addfont(FONT_PATH)
    primary_font = font_manager.FontProperties(fname=FONT_PATH).get_name()
else:
    primary_font = "Malgun Gothic" if "Malgun Gothic" in available_fonts else None

font_candidates = [primary_font, "Malgun Gothic", "NanumGothic", "DejaVu Sans"]
active_fonts = []
for fam in font_candidates:
    if fam and fam in available_fonts and fam not in active_fonts:
        active_fonts.append(fam)

sns.set_theme(style="whitegrid")
if active_fonts:
    rc("font", family=active_fonts)
    matplotlib.rcParams["font.family"] = active_fonts
rc("axes", unicode_minus=False)
matplotlib.rcParams["axes.unicode_minus"] = False

plt.rcParams["figure.dpi"] = 120
print("ready:", NOTEBOOK_DIR)
print("active fonts:", matplotlib.rcParams.get("font.family"))

## 1. 데이터 로딩 및 클러스터 단위 피처 테이블 생성

In [ ]:
DEFAULT_ROOT = NOTEBOOK_DIR / "vod-received" / "view_of_delft_PUBLIC"
DATASET_ROOT = Path(os.environ.get("VOD_ROOT", str(DEFAULT_ROOT))).resolve()

RADAR_MODE = os.environ.get("VOD_RADAR_MODE", "5-scan")
MAX_FRAMES = int(os.environ.get("VOD_RISK_MAX_FRAMES", "180"))
MAX_POINTS_PER_FRAME = int(os.environ.get("VOD_RISK_MAX_POINTS_PER_FRAME", "1800"))

DBSCAN_EPS = float(os.environ.get("VOD_RISK_DBSCAN_EPS", "1.25"))
DBSCAN_MIN_SAMPLES = int(os.environ.get("VOD_RISK_DBSCAN_MIN_SAMPLES", "6"))
LIDAR_VERIFY_RADIUS = float(os.environ.get("VOD_RISK_LIDAR_VERIFY_RADIUS", "1.8"))

frames_all = bev.list_vod_sync_frames(DATASET_ROOT, radar_mode=RADAR_MODE)
frames = [fr for fr in frames_all if fr["radar_path"] and Path(fr["radar_path"]).is_file()][:MAX_FRAMES]
if not frames:
    raise RuntimeError("사용 가능한 radar frame이 없습니다. VOD_ROOT/RADAR_MODE 확인 필요")

frame_order_map = {fr["frame_id"]: i for i, fr in enumerate(frames)}
print("DATASET_ROOT:", DATASET_ROOT)
print("RADAR_MODE:", RADAR_MODE)
print("사용 frame 수:", len(frames), "/", len(frames_all))


def _preprocess_radar_points(pts: np.ndarray) -> np.ndarray:
    if pts.size == 0:
        return np.zeros((0, 7), dtype=np.float32)
    m = (
        (pts[:, 0] >= 0.0)
        & (pts[:, 0] <= 70.0)
        & (pts[:, 1] >= -35.0)
        & (pts[:, 1] <= 35.0)
        & (pts[:, 2] >= -3.5)
        & (pts[:, 2] <= 4.0)
        & (pts[:, 3] >= -40.0)
        & (pts[:, 3] <= 40.0)
    )
    out = pts[m].copy()
    if out.shape[0] > MAX_POINTS_PER_FRAME:
        idx = np.random.default_rng(SEED).choice(out.shape[0], MAX_POINTS_PER_FRAME, replace=False)
        out = out[idx]
    return out


def _cluster_frame(fr: dict[str, Any]) -> pd.DataFrame:
    radar_raw = bev.parse_radar_bin(Path(fr["radar_path"]))
    pts = _preprocess_radar_points(radar_raw)
    if pts.shape[0] == 0:
        return pd.DataFrame()

    xyz = pts[:, :3]
    rcs = pts[:, 3]
    vr_comp = pts[:, 5]

    feat = np.column_stack([xyz[:, 0], xyz[:, 1], vr_comp])
    feat_scaled = RobustScaler().fit_transform(feat)

    db = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
    labels = db.fit_predict(feat_scaled)

    lidar_min_dist_by_cluster: dict[int, float] = {}
    lidar_path = fr.get("lidar_path")
    if lidar_path and Path(lidar_path).is_file():
        lidar_pts = bev.parse_lidar_bin(Path(lidar_path))
        lidar_xy = lidar_pts[:, :2]
        if len(lidar_xy) > 0:
            centers = []
            cids = []
            for cid in np.unique(labels):
                if cid < 0:
                    continue
                cpts = xyz[labels == cid]
                if cpts.shape[0] == 0:
                    continue
                centers.append([float(cpts[:, 0].mean()), float(cpts[:, 1].mean())])
                cids.append(int(cid))
            if centers:
                nn = NearestNeighbors(n_neighbors=1)
                nn.fit(lidar_xy)
                dists, _ = nn.kneighbors(np.asarray(centers))
                lidar_min_dist_by_cluster = {cid: float(d) for cid, d in zip(cids, dists[:, 0])}

    rows = []
    for cid in np.unique(labels):
        if cid < 0:
            continue
        c_mask = labels == cid
        c_xyz = xyz[c_mask]
        c_rcs = rcs[c_mask]
        c_vr_comp = vr_comp[c_mask]
        if c_xyz.shape[0] == 0:
            continue

        cx, cy, cz = c_xyz.mean(axis=0)
        range_xy = float(np.hypot(cx, cy))
        abs_vr_comp = float(np.mean(np.abs(c_vr_comp)))
        spread_xy = float(np.sqrt(np.var(c_xyz[:, 0]) + np.var(c_xyz[:, 1])))
        n_points = int(c_xyz.shape[0])
        density_proxy = float(n_points / max(spread_xy + 1e-3, 1e-3))

        lidar_min_dist = lidar_min_dist_by_cluster.get(int(cid), np.nan)
        lidar_verified = bool(np.isfinite(lidar_min_dist) and lidar_min_dist <= LIDAR_VERIFY_RADIUS)

        rows.append(
            {
                "frame_id": fr["frame_id"],
                "frame_order": int(frame_order_map[fr["frame_id"]]),
                "cluster_id": int(cid),
                "n_points": n_points,
                "cx": float(cx),
                "cy": float(cy),
                "cz": float(cz),
                "range_xy": range_xy,
                "mean_rcs": float(np.mean(c_rcs)),
                "rcs_std": float(np.std(c_rcs)),
                "mean_vr_comp": float(np.mean(c_vr_comp)),
                "abs_vr_comp": abs_vr_comp,
                "vr_comp_std": float(np.std(c_vr_comp)),
                "z_std": float(np.std(c_xyz[:, 2])),
                "spread_xy": spread_xy,
                "density_proxy": density_proxy,
                "lidar_min_dist": float(lidar_min_dist) if np.isfinite(lidar_min_dist) else np.nan,
                "lidar_verified": int(lidar_verified),
            }
        )

    return pd.DataFrame(rows)


cluster_rows = []
for fr in frames:
    cdf = _cluster_frame(fr)
    if not cdf.empty:
        cluster_rows.append(cdf)

if not cluster_rows:
    raise RuntimeError("클러스터 데이터가 비어 있습니다. DBSCAN/ROI 파라미터를 점검하세요.")

cluster_df = pd.concat(cluster_rows, ignore_index=True)

print("cluster_df shape:", cluster_df.shape)
print("frame 수:", cluster_df["frame_id"].nunique())
print("class 준비 전 클러스터 수:", len(cluster_df))
display(cluster_df.head())

## 2. Part A — Rule-based Risk Estimation (운영 baseline)

여기서는 운영 관점의 baseline 규칙을 분리해서 정의합니다.

- `risk_label_rule`: 규칙 기반 위험 등급
- `risk_score_rule`: 규칙 기반 연속 점수

중요:
- `risk_label_rule`은 운영/의사결정 baseline입니다.
- ML 모델은 이 규칙을 그대로 외우는 모델이 되면 안 됩니다.

In [ ]:
def compute_ttc(df: pd.DataFrame) -> pd.Series:
    # approaching 속도의 하한을 둬서 분모 0/폭주를 방지
    speed = df["abs_vr_comp"].clip(lower=0.2)
    ttc = df["range_xy"] / speed
    return ttc.clip(lower=0.0, upper=60.0)


def make_rule_based_risk_label(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["ttc"] = compute_ttc(out)

    # 위험 점수(0~1): 가까움/접근속도/클러스터 밀도/LiDAR 검증을 결합
    range_term = 1.0 - np.clip(out["range_xy"] / 60.0, 0.0, 1.0)
    speed_term = np.clip(out["abs_vr_comp"] / 8.0, 0.0, 1.0)
    ttc_term = np.clip((8.0 - out["ttc"]) / 8.0, 0.0, 1.0)
    density_term = np.clip(out["density_proxy"] / (out["density_proxy"].quantile(0.9) + 1e-6), 0.0, 1.0)
    lidar_term = np.where(out["lidar_verified"] > 0, 0.08, -0.03)

    score = 0.35 * ttc_term + 0.25 * range_term + 0.22 * speed_term + 0.10 * density_term + lidar_term
    out["risk_score_rule"] = np.clip(score, 0.0, 1.0)

    out["risk_label_rule"] = np.where(
        out["risk_score_rule"] >= 0.68,
        "high",
        np.where(out["risk_score_rule"] >= 0.42, "medium", "low"),
    )

    # ML 실험용 타깃을 명시적으로 분리 (혼동 방지)
    out["risk_label_ml_target"] = out["risk_label_rule"].astype(str)
    return out


risk_df = make_rule_based_risk_label(cluster_df)

print("rule label 분포")
display(risk_df["risk_label_rule"].value_counts().to_frame("count"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(risk_df["risk_score_rule"], bins=30, ax=axes[0], color="#4c72b0")
axes[0].set_title("risk_score_rule 분포")
axes[0].set_xlabel("risk_score_rule")

sns.countplot(data=risk_df, x="risk_label_rule", order=["low", "medium", "high"], ax=axes[1], palette="Set2")
axes[1].set_title("risk_label_rule 분포")
axes[1].set_xlabel("risk_label_rule")
plt.tight_layout()
plt.show()

display(risk_df.head())

## 3. Feature Set 구성 (Leakage 방지 vs 진단용)

- **Feature Set A (strict anti-leakage)**: 라벨 규칙과 직접 겹치는 핵심 변수 제외
- **Feature Set B (moderate anti-leakage)**: A + LiDAR 검증 보조 정보
- **Feature Set C (diagnostic only)**: 누설 포함(기존 방식 재현/진단 용도)

반드시 제외(anti-leakage):
- `ttc`
- `range_xy`
- `abs_vr_comp`

> Feature Set C는 누설 진단용으로만 사용하며 최종 채택 대상이 아닙니다.

In [ ]:
def build_feature_sets(df: pd.DataFrame) -> tuple[dict[str, list[str]], pd.DataFrame]:
    # strict anti-leakage
    feat_A = [
        "n_points",
        "spread_xy",
        "mean_rcs",
        "rcs_std",
        "mean_vr_comp",
        "vr_comp_std",
        "z_std",
        "density_proxy",
        "cz",
    ]

    # moderate anti-leakage (LiDAR 보조 추가)
    feat_B = feat_A + ["lidar_verified", "lidar_min_dist"]

    # diagnostic leakage 포함 (최종 채택 금지)
    feat_C = [
        "n_points",
        "spread_xy",
        "mean_rcs",
        "mean_vr_comp",
        "cx",
        "cy",
        "cz",
        "lidar_min_dist",
        "ttc",
        "range_xy",
        "abs_vr_comp",
    ]

    # 실제 존재 컬럼만 남김
    feature_sets = {}
    for name, cols in {
        "A_strict_anti_leakage": feat_A,
        "B_moderate_anti_leakage": feat_B,
        "C_diagnostic_leakage": feat_C,
    }.items():
        feature_sets[name] = [c for c in cols if c in df.columns]

    rows = []
    must_exclude = {"ttc", "range_xy", "abs_vr_comp"}
    for name, cols in feature_sets.items():
        rows.append(
            {
                "feature_set": name,
                "n_features": len(cols),
                "includes_ttc": "ttc" in cols,
                "includes_range_xy": "range_xy" in cols,
                "includes_abs_vr_comp": "abs_vr_comp" in cols,
                "contains_leakage_core": len(must_exclude.intersection(cols)) > 0,
                "features": ", ".join(cols),
            }
        )

    return feature_sets, pd.DataFrame(rows)


feature_sets, feature_set_table = build_feature_sets(risk_df)
display(feature_set_table)

## Part B — ML-based Risk Estimation (보조 추론 실험)

이 파트의 목적은 **규칙을 외우는 모델**을 만드는 것이 아니라,
간접 특징만으로 위험 수준을 어느 정도 추론 가능한지 확인하는 것입니다.

- Rule-based: 운영 baseline
- ML-based: 보조 추론 실험
- 두 성능 수치는 같은 의미가 아니며, 직접 비교 시 해석에 주의해야 합니다.

## 4. 분할/모델/평가 함수 정의

여기서부터는 실험 함수화를 통해 중복을 제거합니다.

필수 함수:
- `split_data(df, split_mode)`
- `build_model(model_name)`
- `run_experiment(feature_set, model_name, split_mode)`
- `plot_confusion_matrix(...)`
- `plot_class_distribution(...)`
- `plot_feature_importance(...)`
- `summarize_experiments(results)`

In [ ]:
TARGET_COL = "risk_label_ml_target"
LABEL_ORDER = ["low", "medium", "high"]


def split_data(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str = TARGET_COL,
    split_mode: str = "random",
    random_state: int = SEED,
    test_size: float = 0.25,
) -> dict[str, Any]:
    work = df.copy()
    work = work.dropna(subset=[target_col]).reset_index(drop=True)

    X = work[feature_cols].replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))
    y = work[target_col].astype(str)

    if split_mode == "random":
        tr_idx, te_idx = train_test_split(
            np.arange(len(work)),
            test_size=test_size,
            random_state=random_state,
            stratify=y,
        )
    elif split_mode == "group":
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        groups = work["frame_id"].astype(str)
        tr_idx, te_idx = next(gss.split(X, y, groups=groups))
    elif split_mode == "time":
        uniq = (
            work[["frame_id", "frame_order"]]
            .drop_duplicates()
            .sort_values("frame_order")
            .reset_index(drop=True)
        )
        n = len(uniq)
        if n < 10:
            raise RuntimeError("time split을 위한 frame 수가 부족합니다")

        tr_end = max(int(n * 0.70), 1)
        val_end = max(int(n * 0.85), tr_end + 1)
        train_frames = set(uniq.iloc[:tr_end]["frame_id"].astype(str))
        test_frames = set(uniq.iloc[val_end:]["frame_id"].astype(str))

        tr_idx = np.where(work["frame_id"].astype(str).isin(train_frames))[0]
        te_idx = np.where(work["frame_id"].astype(str).isin(test_frames))[0]
    else:
        raise ValueError(f"unknown split_mode: {split_mode}")

    X_train, X_test = X.iloc[tr_idx].copy(), X.iloc[te_idx].copy()
    y_train, y_test = y.iloc[tr_idx].copy(), y.iloc[te_idx].copy()

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "train_idx": tr_idx,
        "test_idx": te_idx,
        "split_mode": split_mode,
    }


def build_model(model_name: str, random_state: int = SEED):
    name = model_name.lower()
    if name == "logistic_regression":
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=2500,
                        class_weight="balanced",
                        random_state=random_state,
                    ),
                ),
            ]
        )
    if name == "random_forest":
        return RandomForestClassifier(
            n_estimators=320,
            max_depth=10,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=random_state,
            n_jobs=-1,
        )
    if name == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(
            learning_rate=0.07,
            max_depth=8,
            max_iter=260,
            random_state=random_state,
        )
    if name == "svm_rbf":
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "clf",
                    SVC(
                        kernel="rbf",
                        C=3.0,
                        gamma="scale",
                        class_weight="balanced",
                        probability=True,
                    ),
                ),
            ]
        )
    raise ValueError(f"unknown model_name: {model_name}")


def _fit_with_sample_weight(model, X_train, y_train, sample_weight):
    # estimator마다 sample_weight 지원 방식이 달라 예외를 순차 처리
    try:
        model.fit(X_train, y_train, sample_weight=sample_weight)
        return model
    except Exception:
        pass

    if isinstance(model, Pipeline):
        last_step = model.steps[-1][0]
        try:
            model.fit(X_train, y_train, **{f"{last_step}__sample_weight": sample_weight})
            return model
        except Exception:
            pass

    model.fit(X_train, y_train)
    return model


def run_experiment(
    df: pd.DataFrame,
    feature_set_name: str,
    feature_cols: list[str],
    model_name: str,
    split_mode: str,
    random_state: int = SEED,
) -> dict[str, Any]:
    split_out = split_data(
        df=df,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        split_mode=split_mode,
        random_state=random_state,
    )

    X_train = split_out["X_train"].copy()
    X_test = split_out["X_test"].copy()
    y_train = split_out["y_train"].copy()
    y_test = split_out["y_test"].copy()

    # SVM 계산량 보호
    if model_name == "svm_rbf" and len(X_train) > 9000:
        keep = np.random.default_rng(random_state).choice(len(X_train), size=9000, replace=False)
        X_train = X_train.iloc[keep]
        y_train = y_train.iloc[keep]

    model = build_model(model_name, random_state=random_state)

    sw = compute_sample_weight(class_weight="balanced", y=y_train)
    model = _fit_with_sample_weight(model, X_train, y_train, sw)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred_test, labels=LABEL_ORDER)

    report = classification_report(
        y_test,
        y_pred_test,
        labels=LABEL_ORDER,
        output_dict=True,
        zero_division=0,
    )

    return {
        "feature_set": feature_set_name,
        "split_mode": split_mode,
        "model_name": model_name,
        "n_features": len(feature_cols),
        "feature_cols": feature_cols,
        "train_size": int(len(X_train)),
        "test_size": int(len(X_test)),
        "train_accuracy": float(accuracy_score(y_train, y_pred_train)),
        "test_accuracy": float(accuracy_score(y_test, y_pred_test)),
        "test_macro_f1": float(f1_score(y_test, y_pred_test, average="macro", zero_division=0)),
        "test_weighted_f1": float(f1_score(y_test, y_pred_test, average="weighted", zero_division=0)),
        "classification_report": report,
        "confusion_matrix": cm,
        "model": model,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "y_pred_test": y_pred_test,
        "split_out": split_out,
    }


def plot_confusion_matrix(cm: np.ndarray, labels: list[str], title: str):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Pred")
    ax.set_ylabel("True")
    plt.tight_layout()
    plt.show()


def plot_class_distribution(y_train: pd.Series, y_test: pd.Series, title: str):
    tr = y_train.value_counts().reindex(LABEL_ORDER).fillna(0)
    te = y_test.value_counts().reindex(LABEL_ORDER).fillna(0)
    dist_df = pd.DataFrame({"train": tr, "test": te})

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    sns.barplot(x=dist_df.index, y=dist_df["train"].values, ax=axes[0], palette="Blues")
    axes[0].set_title(f"{title} - train class dist")
    sns.barplot(x=dist_df.index, y=dist_df["test"].values, ax=axes[1], palette="Oranges")
    axes[1].set_title(f"{title} - test class dist")
    plt.tight_layout()
    plt.show()


def plot_feature_importance(importances: pd.Series, title: str, top_n: int = 12):
    imp = importances.sort_values(ascending=False).head(top_n)[::-1]
    fig, ax = plt.subplots(figsize=(7, max(4, len(imp) * 0.32)))
    ax.barh(imp.index, imp.values, color="#4c72b0")
    ax.set_title(title)
    ax.set_xlabel("importance")
    plt.tight_layout()
    plt.show()


def summarize_experiments(results: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for r in results:
        rows.append(
            {
                "feature_set": r["feature_set"],
                "split_mode": r["split_mode"],
                "model_name": r["model_name"],
                "train_size": r["train_size"],
                "test_size": r["test_size"],
                "train_accuracy": round(r["train_accuracy"], 4),
                "test_accuracy": round(r["test_accuracy"], 4),
                "test_macro_f1": round(r["test_macro_f1"], 4),
                "test_weighted_f1": round(r["test_weighted_f1"], 4),
                "overfit_gap_acc": round(r["train_accuracy"] - r["test_accuracy"], 4),
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["test_macro_f1", "test_accuracy"], ascending=[False, False]
    ).reset_index(drop=True)

## 5. 알고리즘 구성 이유 (단순 + 비교적 고급)

- **LogisticRegression**
  - 가장 단순한 선형 baseline
  - feature와 label 관계가 거의 선형인지 빠르게 점검 가능

- **RandomForestClassifier**
  - 비선형/임계값 기반 규칙을 잘 처리
  - tabular baseline으로 강하고 해석(importance)도 비교적 쉬움

- **HistGradientBoostingClassifier**
  - 구조화(tabular) 데이터에서 강한 고급 모델
  - 복잡한 feature interaction과 비선형 경계 학습에 유리

- **SVM (RBF)**
  - 복잡한 결정경계 학습 가능
  - 샘플 수가 커질수록 계산량 부담이 커질 수 있어 실무에서는 제약 존재

## 6. 기존 방식 재현: 누설 포함 + random split

이 섹션은 의도적으로 누설이 있는 Feature Set C를 사용해, 왜 성능이 비정상적으로 높아질 수 있는지 진단합니다.

In [ ]:
baseline_results = []
for m in ["logistic_regression", "random_forest", "hist_gradient_boosting", "svm_rbf"]:
    res = run_experiment(
        df=risk_df,
        feature_set_name="C_diagnostic_leakage",
        feature_cols=feature_sets["C_diagnostic_leakage"],
        model_name=m,
        split_mode="random",
    )
    baseline_results.append(res)

baseline_table = summarize_experiments(baseline_results)
print("[누설 포함 + random split] 결과")
display(baseline_table)

# 대표 1개 confusion matrix
best_baseline = baseline_results[np.argmax([r["test_macro_f1"] for r in baseline_results])]
plot_class_distribution(best_baseline["y_train"], best_baseline["y_test"], "Leakage+Random")
plot_confusion_matrix(
    best_baseline["confusion_matrix"],
    LABEL_ORDER,
    f"Leakage+Random | {best_baseline['model_name']}"
)

print("best baseline classification report")
display(pd.DataFrame(best_baseline["classification_report"]).T)

## 7. 개선 방식 비교 실험 (최소 8개)

아래 실험은 random/group/time split과 A/B/C feature set을 교차 비교합니다.

- C는 누설 진단용
- A/B는 실제 anti-leakage 비교용

In [ ]:
EXPERIMENT_PLAN = [
    # leakage 진단
    ("C_diagnostic_leakage", "logistic_regression", "random", "누설+랜덤에서 선형 baseline도 과대평가되는지 확인"),
    ("C_diagnostic_leakage", "random_forest", "random", "누설+랜덤에서 트리 모델 과대성능 확인"),
    ("C_diagnostic_leakage", "random_forest", "group", "누설 feature만 남기고 split만 바꿨을 때 변화 확인"),

    # anti-leakage 핵심 비교
    ("A_strict_anti_leakage", "random_forest", "random", "feature 누설 제거만으로 random split 점수 하락 여부 확인"),
    ("A_strict_anti_leakage", "random_forest", "group", "핵심 비교: anti-leakage + group"),
    ("A_strict_anti_leakage", "logistic_regression", "group", "단순 선형 기준선"),
    ("A_strict_anti_leakage", "hist_gradient_boosting", "group", "고급 모델의 비선형 학습 비교"),
    ("A_strict_anti_leakage", "svm_rbf", "group", "복잡 경계 모델 비교"),
    ("A_strict_anti_leakage", "hist_gradient_boosting", "time", "시간 일반화(미래 프레임) 강건성 확인"),

    # moderate anti-leakage
    ("B_moderate_anti_leakage", "random_forest", "group", "LiDAR 보조 정보 추가 효과 확인"),
    ("B_moderate_anti_leakage", "hist_gradient_boosting", "group", "고급 모델 + LiDAR 보조 정보"),
    ("B_moderate_anti_leakage", "svm_rbf", "time", "시간 분할에서 복잡 모델 안정성 확인"),
]

plan_df = pd.DataFrame(
    EXPERIMENT_PLAN,
    columns=["feature_set", "model_name", "split_mode", "검증 목적"],
)
print("실험 계획표")
display(plan_df)

all_results = []
for fs_name, model_name, split_mode, _ in EXPERIMENT_PLAN:
    print(f"run: {fs_name} | {model_name} | {split_mode}")
    res = run_experiment(
        df=risk_df,
        feature_set_name=fs_name,
        feature_cols=feature_sets[fs_name],
        model_name=model_name,
        split_mode=split_mode,
    )
    all_results.append(res)

results_table = summarize_experiments(all_results)
print("\n[전체 실험 요약]")
display(results_table)

pivot_macro = results_table.pivot_table(
    index=["feature_set", "split_mode"],
    columns="model_name",
    values="test_macro_f1",
    aggfunc="mean",
)
print("[macro F1 pivot]")
display(pivot_macro)

# 누설 없는 조건(A/B + group/time)에서 상위 결과
honest_table = results_table[
    (results_table["feature_set"].isin(["A_strict_anti_leakage", "B_moderate_anti_leakage"]))
    & (results_table["split_mode"].isin(["group", "time"]))
].copy()
print("[정직한 평가 후보 상위]")
display(honest_table.sort_values("test_macro_f1", ascending=False).head(10))

## 8. 진단 시각화: random vs group vs time, leakage vs anti-leakage

accuracy뿐 아니라 macro F1, confusion matrix, class 분포를 함께 비교합니다.

- accuracy 단독 해석은 클래스 불균형에서 착시가 발생할 수 있음
- macro F1은 클래스 균형 관점에서 더 중요함

In [ ]:
def _pick_result(results: list[dict[str, Any]], fs: str, split: str, model: str):
    for r in results:
        if r["feature_set"] == fs and r["split_mode"] == split and r["model_name"] == model:
            return r
    return None

sample_cases = [
    ("C_diagnostic_leakage", "random", "random_forest"),
    ("C_diagnostic_leakage", "group", "random_forest"),
    ("A_strict_anti_leakage", "group", "random_forest"),
    ("A_strict_anti_leakage", "time", "hist_gradient_boosting"),
]

for fs, sp, md in sample_cases:
    r = _pick_result(all_results, fs, sp, md)
    if r is None:
        continue

    title = f"{fs} | {sp} | {md}"
    print("\n", title)
    print(
        {
            "train_acc": round(r["train_accuracy"], 4),
            "test_acc": round(r["test_accuracy"], 4),
            "test_macro_f1": round(r["test_macro_f1"], 4),
            "test_weighted_f1": round(r["test_weighted_f1"], 4),
        }
    )
    plot_class_distribution(r["y_train"], r["y_test"], title)
    plot_confusion_matrix(r["confusion_matrix"], LABEL_ORDER, title)

    print("per-class report")
    display(pd.DataFrame(r["classification_report"]).T.loc[LABEL_ORDER + ["macro avg", "weighted avg"]])

## 9. Feature Importance / Permutation Importance / (가능 시) SHAP

분석 기준:
- anti-leakage + group split 조건에서 성능이 높은 모델을 기준으로 해석
- 누설 feature 제거 후에도 어떤 특징이 기여하는지 확인

In [ ]:
# 해석 대상: anti-leakage(A/B) + group split 중 macro F1 최고 모델
cand = [
    r for r in all_results
    if r["feature_set"] in ["A_strict_anti_leakage", "B_moderate_anti_leakage"]
    and r["split_mode"] == "group"
]
if not cand:
    raise RuntimeError("importance 분석 대상 실험이 없습니다.")

best_honest = sorted(cand, key=lambda x: x["test_macro_f1"], reverse=True)[0]
print("importance 대상:", best_honest["feature_set"], best_honest["split_mode"], best_honest["model_name"])
print("test_macro_f1:", round(best_honest["test_macro_f1"], 4))

model = best_honest["model"]
X_test = best_honest["X_test"]
y_test = best_honest["y_test"]
feature_cols = best_honest["feature_cols"]

# 1) model 기반 importance
imp_series = None
if hasattr(model, "feature_importances_"):
    imp_series = pd.Series(model.feature_importances_, index=feature_cols)
elif isinstance(model, Pipeline) and hasattr(model.steps[-1][1], "feature_importances_"):
    imp_series = pd.Series(model.steps[-1][1].feature_importances_, index=feature_cols)

if imp_series is not None:
    plot_feature_importance(imp_series, "Model Feature Importance")
else:
    print("이 모델은 feature_importances_를 직접 제공하지 않습니다.")

# 2) permutation importance
perm = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=8,
    random_state=SEED,
    n_jobs=1,
)
perm_series = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)
plot_feature_importance(perm_series, "Permutation Importance (macro F1)")

# 3) SHAP (환경 가능 시)
try:
    import shap

    if isinstance(model, RandomForestClassifier):
        explainer = shap.TreeExplainer(model)
        x_small = X_test.sample(min(300, len(X_test)), random_state=SEED)
        shap_values = explainer.shap_values(x_small)
        print("SHAP summary (RandomForest) 실행")
        shap.summary_plot(shap_values, x_small, show=True)
    elif isinstance(model, Pipeline) and isinstance(model.steps[-1][1], RandomForestClassifier):
        est = model.steps[-1][1]
        x_small = X_test.sample(min(300, len(X_test)), random_state=SEED)
        # Pipeline의 scaler 출력 공간과 원본 공간 차이 때문에 여기서는 permutation 권장
        print("Pipeline+RF에서는 permutation importance를 우선 해석 지표로 사용")
    else:
        print("현재 best 모델 유형에서 SHAP을 생략하고 permutation importance를 사용합니다.")
except Exception as e:
    print("SHAP 생략:", str(e)[:200])

## 9-1. 왜 `accuracy 1.000`이 나왔는지 정량 진단

In [ ]:
diag_rows = []
for r in all_results:
    diag_rows.append(
        {
            "case": f"{r['feature_set']} | {r['split_mode']} | {r['model_name']}",
            "feature_set": r["feature_set"],
            "split_mode": r["split_mode"],
            "model_name": r["model_name"],
            "test_accuracy": r["test_accuracy"],
            "test_macro_f1": r["test_macro_f1"],
            "overfit_gap_acc": r["train_accuracy"] - r["test_accuracy"],
        }
    )

diag_df = pd.DataFrame(diag_rows)

d_leak_random = diag_df[
    (diag_df["feature_set"] == "C_diagnostic_leakage")
    & (diag_df["split_mode"] == "random")
]
d_honest = diag_df[
    (diag_df["feature_set"].isin(["A_strict_anti_leakage", "B_moderate_anti_leakage"]))
    & (diag_df["split_mode"].isin(["group", "time"]))
]

summary_diag = pd.DataFrame(
    {
        "leakage_random_acc_mean": [d_leak_random["test_accuracy"].mean()],
        "leakage_random_macro_f1_mean": [d_leak_random["test_macro_f1"].mean()],
        "honest_acc_mean": [d_honest["test_accuracy"].mean()],
        "honest_macro_f1_mean": [d_honest["test_macro_f1"].mean()],
        "acc_drop(leak->honest)": [d_leak_random["test_accuracy"].mean() - d_honest["test_accuracy"].mean()],
        "macro_f1_drop(leak->honest)": [d_leak_random["test_macro_f1"].mean() - d_honest["test_macro_f1"].mean()],
    }
)

print("[진단 요약] 누설+random 대비 정직 평가(anti-leakage+group/time) 성능 변화")
display(summary_diag.round(4))

print("[상세 진단표]")
display(diag_df.sort_values(["test_macro_f1", "test_accuracy"], ascending=False).round(4))

## 10. 발표용 핵심 해석 (한국어)

### 1) 기존 accuracy 1.000은 왜 신뢰하면 안 되는가
- 라벨(`risk_label_rule`) 생성에 쓰인 핵심 변수(`ttc`, `range_xy`, `abs_vr_comp`)가 모델 입력에 들어가면, 모델은 일반화가 아니라 규칙 복원을 수행할 수 있습니다.
- 특히 random split에서는 인접 프레임의 유사 샘플이 train/test에 함께 들어가 성능이 추가로 부풀려집니다.

### 2) 라벨 생성 규칙과 입력 특징 중복의 문제
- 규칙 기반 타깃을 만들고 동일 규칙 변수를 입력으로 넣으면 구조적 leakage가 발생합니다.
- 이 경우 고급 모델일수록 규칙을 더 완벽히 암기해 겉보기 점수가 더 좋아질 수 있어 위험합니다.

### 3) random split이 프레임 데이터에서 위험한 이유
- 동일 장면의 연속 프레임 샘플은 매우 유사합니다.
- random split은 독립성 가정을 깨뜨려 실전보다 쉬운 테스트셋을 만들 가능성이 큽니다.

### 4) group/time split이 더 현실적인 이유
- group split(frame_id 기준)은 프레임 단위 중복 누출을 줄입니다.
- time split(앞 70% 학습, 뒤 15% 테스트)은 미래 프레임 일반화 관점에서 더 엄격합니다.

### 5) 단순 모델 vs 고급 모델 차이
- LogisticRegression: 선형 기준선으로 최소 성능 하한/데이터 선형성 확인에 유리
- RandomForest: 비선형 규칙 처리에 강하고 해석이 쉬운 실무 baseline
- HistGradientBoosting: tabular에서 강한 고급 모델로 복잡 상호작용 포착
- SVM(RBF): 복잡 경계 학습 가능하나 대규모 데이터에서 계산비용 부담

### 6) 고급 모델이라고 항상 좋은 것은 아님
- 누설 구조에서는 고급 모델이 더 높은 점수로 더 잘 속일 수 있습니다.
- 모델 복잡도보다 **평가 구조의 정직성(split/feature 설계)**이 우선입니다.

### 7) 개선 후 결과를 왜 더 신뢰할 수 있는가
- anti-leakage feature(A/B) + group/time split으로 누설 가능성을 줄였고,
- accuracy뿐 아니라 macro F1, confusion matrix, class 분포를 함께 확인했습니다.

### 8) 현재 ML 결과의 위치
- 현재 ML 결과는 외부 GT 기반 최종 검출 모델이 아니라 **rule-based 위험도 보조 추론 실험**입니다.

### 9) 운영 관점 정리
- **Rule-based risk = 운영 baseline**
- **ML-based risk = 보조 추론/우선순위화 도구**
- 발표 시 두 결과를 동일 의미의 점수로 혼용하지 않도록 분리 설명해야 합니다.

## 11. 최종 결론 템플릿 (발표/PPT 복사용)

- 가장 정직한 평가는 `A/B anti-leakage + group/time split + macro F1` 기준입니다.
- `C leakage + random split`의 고성능은 실전 일반화가 아니라 규칙 복원/분할 이슈 가능성을 먼저 의심해야 합니다.
- 모델 선택은 단순히 최고 accuracy가 아니라:
  1) split 강건성(random↔group↔time 편차),
  2) class별 성능 균형(macro F1),
  3) 해석 가능성(importance/permutation)
  을 함께 보아야 합니다.
- 운영 파이프라인에서는 rule-based를 baseline으로 유지하고, ML은 보조 추론/우선순위화 용도로 단계적으로 통합하는 것이 안전합니다.